In [32]:
#Importing Spark
import pyspark

In [33]:
#Importing SparkSession
from pyspark.sql import SparkSession

In [34]:
#Importing functions to use on the data set
from pyspark.sql import functions as F

In [35]:
#Initiating a Spark Session
SoccerSession = SparkSession.builder.appName('SoccerSession').getOrCreate();

In [62]:
#Importing the Fifa 2022 Player data from my local system
fifaRaw = SoccerSession.read.csv('C:/Users/0106o/Downloads/Datasets/Fifa Dataset/game_player_stats.csv', inferSchema = True, header = True);

In [63]:
#Importing the World Cup 2022 Player data from my local system
worldCupRaw = SoccerSession.read.csv('C:/Users/0106o/Downloads/Datasets/Fifa Dataset/player_stats.csv', inferSchema = True, header = True);

In [58]:
#Checking the schema to see if there are any column types I want to change
worldCupRaw.printSchema();


root
 |-- player: string (nullable = true)
 |-- position: string (nullable = true)
 |-- team: string (nullable = true)
 |-- age: string (nullable = true)
 |-- club: string (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- games: integer (nullable = true)
 |-- games_starts: integer (nullable = true)
 |-- minutes: integer (nullable = true)
 |-- minutes_90s: double (nullable = true)
 |-- goals: integer (nullable = true)
 |-- assists: integer (nullable = true)
 |-- goals_pens: integer (nullable = true)
 |-- pens_made: integer (nullable = true)
 |-- pens_att: integer (nullable = true)
 |-- cards_yellow: integer (nullable = true)
 |-- cards_red: integer (nullable = true)
 |-- goals_per90: double (nullable = true)
 |-- assists_per90: double (nullable = true)
 |-- goals_assists_per90: double (nullable = true)
 |-- goals_pens_per90: double (nullable = true)
 |-- goals_assists_pens_per90: double (nullable = true)
 |-- xg: double (nullable = true)
 |-- npxg: double (nullable = tru

In [64]:
#Filtering out the players that may not have played in the 2022, World Cup despite being on the team
worldCupRaw = worldCupRaw.filter(worldCupRaw['minutes'] > 0)

In [65]:
#Selecting only the columns I need from the World Cup dataset
columns_to_keep = [
    "player", "position", "team", "minutes", "minutes_90s",
    "goals", "assists", "goals_per90", "assists_per90",
    "goals_assists_pens_per90", "xg", "npxg_per90", "xg_assist_per90",

]

worldCupRaw = worldCupRaw.select(columns_to_keep)

In [66]:
#Selecting only the columns I need from the Fifa 22 dataset
columns_to_keep = ["long_name", "value_eur"]

fifaRaw = fifaRaw.select(columns_to_keep)

In [67]:
#Renaming the name column to match the other dataset
newColumnNames = {"long_name": "player"}
fifaRaw = fifaRaw.withColumnsRenamed(newColumnNames)

In [68]:
#Creating new columns required for joins to the lower case and then trimming for whitespaces
#I don't use the original columns to maintain their integrity for analysis

worldCupRaw = worldCupRaw.withColumn("join_name", F.lower(F.trim(worldCupRaw['player'])))

fifaRaw = fifaRaw.withColumn("join_name", F.lower(F.trim(fifaRaw['player'])))

In [69]:
#Joining the two tables to create the final dataset
finalDataset = worldCupRaw.join( 
    fifaRaw,
    on=["join_name"],
    how="left"
)

In [70]:
total = finalDataset.count()
matched = finalDataset.filter(finalDataset['value_eur'].isNotNull()).count()
print(f"Matched: {matched}/{total} ({round(matched/total*100, 1)}%)")

Matched: 233/682 (34.2%)
